In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: LBCO, HRPT

This basic example is designed to show how Rietveld refinement can be
performed when both the crystal structure and experiment parameters
are defined using CIF files.

For this example, constant-wavelength neutron powder diffraction data
for La0.5Ba0.5CoO3 from HRPT at PSI is used.

The example is intended for users who are already familiar with the
EasyDiffraction library and want to quickly get started with a basic
refinement.

It is also useful for those who want to see how constraints can be
applied to highly correlated parameters. For a more detailed
explanation of the code, please refer to the other tutorials.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

In [3]:
# Create a minimal project with a short name
project = edi.Project(name='lbco_hrpt')

## 🧩 Define Structure

In [4]:
# Download CIF file from repository
structure_path = edi.download_data('struct-lbco', destination='data')

Getting data...


Data 'struct-lbco': La0.5Ba0.5CoO3 (crystal structure)


✅ Data 'struct-lbco' already present at '../../../data/struct-lbco.cif'. Keeping existing.


In [5]:
# Add structure from downloaded CIF
project.structures.add_from_cif_path(structure_path)

In [6]:
# Plot the crystal structure
project.display.structure(struct_name='lbco')

Structure 🧩 'lbco' (Atom view type: 'covalent')


## 🔬 Define Experiment

In [7]:
# Download CIF file from repository
expt_path = edi.download_data('expt-lbco-hrpt', destination='data')

Getting data...


Data 'expt-lbco-hrpt': La0.5Ba0.5CoO3, HRPT (PSI), 300 K


✅ Data 'expt-lbco-hrpt' downloaded to '../../../data/expt-lbco-hrpt.edi'


In [8]:
# Add experiment from downloaded CIF
project.experiments.add_from_cif_path(expt_path)

## 🚀 Perform Analysis

### Without Constraints

In [9]:
# Start refinement. All parameters, which have standard uncertainties
# in the input CIF files, are refined by default.
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.31,165.50,
2,28,1.36,33.67,79.7% ↓
3,45,2.30,10.85,67.8% ↓
4,63,3.03,6.43,40.7% ↓
5,81,3.75,3.33,48.2% ↓
6,98,4.43,2.23,33.2% ↓
7,116,5.19,1.91,14.5% ↓
8,133,5.89,1.50,21.1% ↓
9,150,6.54,1.45,3.6% ↓
10,167,7.21,1.34,7.7% ↓


🏆 Best goodness-of-fit (reduced χ²) is 1.29 at iteration 261


✅ Fitting complete.


In [10]:
# Show fit results summary
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),11.91
4,🔁 Iterations,273
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.62
7,"📏 R-factor squared (Rf², %)",5.25
8,"📏 Weighted R-factor (wR, %)",4.40


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8800,3.8909,0.0000,0.28 % ↑
2,lbco,atom_site,La,adp_iso,Å²,0.5000,0.5051,6011.8674,1.02 % ↑
3,lbco,atom_site,Ba,adp_iso,Å²,0.5000,0.5052,9770.9689,1.04 % ↑
4,lbco,atom_site,Co,adp_iso,Å²,0.5000,0.2371,0.0611,52.59 % ↓
5,lbco,atom_site,O,adp_iso,Å²,0.5000,1.3935,0.0168,178.70 % ↑
6,hrpt,linked_structure,lbco,scale,,10.0000,9.1349,0.0643,8.65 % ↓
7,hrpt,peak,,broad_gauss_u,deg²,0.1000,0.0816,0.0031,18.44 % ↓
8,hrpt,peak,,broad_gauss_v,deg²,-0.1000,-0.1159,0.0067,15.90 % ↑
9,hrpt,peak,,broad_gauss_w,deg²,0.1000,0.1204,0.0033,20.45 % ↑
10,hrpt,peak,,broad_lorentz_y,deg,0.1000,0.0844,0.0021,15.57 % ↓


In [11]:
# Show parameter correlations
project.display.fit.correlations()

### With Constraints

In [12]:
# As can be seen from the parameter-correlation plot, the isotropic
# displacement parameters of La and Ba are highly correlated. Because
# La and Ba share the same mixed-occupancy site, their contributions to
# the neutron diffraction pattern are difficult to separate, especially
# since their coherent scattering lengths are not very different.
# Therefore, it is necessary to constrain them to be equal. First we
# define aliases and then use them to create a constraint.
project.analysis.aliases.create(
    id='biso_La',
    param=project.structures['lbco'].atom_sites['La'].adp_iso,
)
project.analysis.aliases.create(
    id='biso_Ba',
    param=project.structures['lbco'].atom_sites['Ba'].adp_iso,
)
project.analysis.constraints.create(expression='biso_Ba = biso_La')

In [13]:
# Start refinement. All parameters, which have standard uncertainties
# in the input CIF files, are refined by default.
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'hrpt' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.04,1.29,
2,20,0.91,1.29,


🏆 Best goodness-of-fit (reduced χ²) is 1.29 at iteration 19


✅ Fitting complete.


In [14]:
# Show fit results summary
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),0.91
4,🔁 Iterations,17
5,📏 Goodness-of-fit (reduced χ²),1.29
6,"📏 R-factor (Rf, %)",5.62
7,"📏 R-factor squared (Rf², %)",5.25
8,"📏 Weighted R-factor (wR, %)",4.40


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,lbco,cell,,length_a,Å,3.8909,3.8909,0.0000,0.00 % ↑
2,lbco,atom_site,La,adp_iso,Å²,0.5051,0.5051,0.0278,0.00 % ↑
3,lbco,atom_site,Co,adp_iso,Å²,0.2371,0.2371,0.0564,0.00 % ↓
4,lbco,atom_site,O,adp_iso,Å²,1.3935,1.3935,0.0160,0.00 % ↑
5,hrpt,linked_structure,lbco,scale,,9.1349,9.1349,0.0538,0.00 % ↓
6,hrpt,peak,,broad_gauss_u,deg²,0.0816,0.0816,0.0031,0.00 % ↑
7,hrpt,peak,,broad_gauss_v,deg²,-0.1159,-0.1159,0.0066,0.01 % ↑
8,hrpt,peak,,broad_gauss_w,deg²,0.1204,0.1204,0.0032,0.00 % ↑
9,hrpt,peak,,broad_lorentz_y,deg,0.0844,0.0844,0.0021,0.00 % ↓
10,hrpt,instrument,,twotheta_offset,deg,0.6226,0.6226,0.0010,0.00 % ↑


In [15]:
# Show parameter correlations
project.display.fit.correlations()

In [16]:
# Show defined experiment names
project.experiments.show_names()

Defined experiments 🔬


['hrpt']


In [17]:
# Plot measured vs. calculated diffraction patterns
project.display.pattern(expt_name='hrpt')

## 💾 Save Project

In [18]:
project.save_as(dir_path='projects/refine-lbco-hrpt-from-cif')

Saving project 📦 'lbco_hrpt' to '../../../projects/refine-lbco-hrpt-from-cif'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 lbco.edi


├── 📁 experiments/


│   └── 📄 hrpt.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 lbco_hrpt.html
